In [ ]:
#| default_exp build

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory
import subprocess
from kavacha.spec import App

Build an application bundle with the required interpreter, pinned environment, and source stamp.

In [ ]:
#| export
from __future__ import annotations
import json, os, shutil, subprocess, sys, time
from fastcore.all import Path

from kavacha.probe import framework_python, is_framework, py_version, running_from
from kavacha.bundle import finish

In [ ]:
#| export
STAMP = 'build.json'
REEXEC = 'KAVACHA_BUILD_VENV'

def run(*args, **kw):
    print('·', ' '.join(str(a) for a in args))
    return subprocess.run([str(a) for a in args], check=True, **kw)

`STAMP` names the bundle's source stamp. `read_stamp` returns it or `None` when absent or invalid.

In [ ]:
run('echo', 'built')

· echo built


CompletedProcess(args=['echo', 'built'], returncode=0)

In [ ]:
#| hide
test_fail(lambda: run('false'), contains='non-zero')

· false


In [ ]:
#| export
def lock_requirements(root, extras=(), out=None):
    "Export `uv.lock` as pinned requirements, or return `None`."
    root = Path(root)
    if not (root/'uv.lock').exists(): return None
    args = ['uv', 'export', '--frozen', '--no-dev', '--no-hashes', '--no-emit-project',
            '--format', 'requirements-txt']
    for e in extras: args += ['--extra', e]
    try: got = subprocess.run(args, cwd=str(root), capture_output=True, text=True, timeout=300)
    except (OSError, subprocess.SubprocessError): return None
    if got.returncode or not got.stdout.strip():
        print(f'· uv export failed; resolving from pyproject instead: {got.stderr.strip()[:200]}')
        return None
    out = Path(out or root/'packaging'/'.app-venv'/'app-requirements.txt')
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(got.stdout)
    return out

`lock_requirements` exports a pinned requirements file from `uv.lock`, or returns `None` when unavailable.

In [ ]:
tmp = TemporaryDirectory(); root = Path(tmp.name)
lock_requirements(root) is None

True

In [ ]:
#| hide
(root/'uv.lock').write_text('not toml')
test_is(lock_requirements(root), None)
assert not (root/'packaging').exists(), 'nothing written when there is nothing to write'

· uv export failed; resolving from pyproject instead: error: No `pyproject.toml` found in current directory or any parent directory


In [ ]:
#| export
def build_venv(python, venv, root, extras=(), force=False):
    "A plain build venv using `python`."
    venv = Path(venv)
    if force: shutil.rmtree(venv, ignore_errors=True)
    exe = venv/'bin'/'python'
    if exe.exists() and py_version(exe) != py_version(python):
        print(f'· rebuilding {venv.name}: it is on {py_version(exe)}, this build wants {py_version(python)}')
        shutil.rmtree(venv, ignore_errors=True)
    if not exe.exists():
        venv.parent.mkdir(parents=True, exist_ok=True)
        run(python, '-m', 'venv', venv)
    run(exe, '-m', 'pip', 'install', '--upgrade', '--quiet', 'pip', 'setuptools', 'wheel')
    spec = f'{root}[{",".join(extras)}]' if extras else str(root)
    if (req := lock_requirements(root, extras, venv/'app-requirements.txt')) is not None:
        run(exe, '-m', 'pip', 'install', '--quiet', '--upgrade', '-r', req)
        run(exe, '-m', 'pip', 'install', '--quiet', '--no-deps', '-e', spec)
    else: run(exe, '-m', 'pip', 'install', '--quiet', '-e', spec)
    return exe

`build_venv` creates a plain virtual environment from the selected interpreter and installs the pinned build requirements.

In [ ]:
#| export
def git_stamp(root, version=''):
    "The commit the tree is on and whether it is dirty, or `None` outside a checkout."
    def git(*a):
        r = subprocess.run(['git', *a], cwd=str(root), capture_output=True, text=True, timeout=30)
        return r.stdout.strip() if not r.returncode else None
    try: sha = git('rev-parse', 'HEAD')
    except (OSError, subprocess.SubprocessError): return None
    if not sha: return None
    return {'commit': sha, 'dirty': bool(git('status', '--porcelain', '--untracked-files=no')),
            'version': version, 'built': time.strftime('%Y-%m-%dT%H:%M:%S')}

def stamp_path(bundle):
    "Where the build stamp lives inside `bundle`, on either platform."
    b = Path(bundle)
    return (b/'Contents'/'Resources'/STAMP) if b.suffix == '.app' else (b/STAMP)

def write_stamp(bundle, root, version=''):
    "Write and return the bundle's source stamp."
    if (st := git_stamp(root, version)) is None:
        st = {'commit': '', 'dirty': False, 'version': version,
              'built': time.strftime('%Y-%m-%dT%H:%M:%S')}
    p = stamp_path(bundle)
    if p.parent.is_dir(): p.write_text(json.dumps(st, indent=1) + '\n')
    return st

def read_stamp(bundle):
    "The stamp `bundle` was built with, or None when it has none."
    try: return json.loads(stamp_path(bundle).read_text())
    except (OSError, ValueError): return None

A bundle is a copy of a tree with nothing pointing back at it. The stamp is the pointer back.

`stamp_path` puts it inside `Contents/Resources` for a `.app` and beside the executable for
anything else. The `.app` suffix is the whole test, so a path names a place before either exists.

In [ ]:
stamp_path('/dist/Demo.app'), stamp_path('/dist/Demo')

(Path('/dist/Demo.app/Contents/Resources/build.json'),
 Path('/dist/Demo/build.json'))

`git_stamp` returns commit, dirty state, version, and build time, or `None` outside a Git checkout.

In [ ]:
#| hide
def mkrepo(d):
    "A checkout with one commit in it, which is what a stamp reads."
    def git(*a): subprocess.run(['git', *a], cwd=d, capture_output=True, check=True)
    git('init', '-q'); git('config', 'user.email', 'a@b.c'); git('config', 'user.name', 'T')
    (Path(d)/'x.txt').write_text('one'); git('add', '-A'); git('commit', '-qm', 'first')
    return Path(d)
t2 = TemporaryDirectory(); repo = mkrepo(t2.name)
(repo/'dist'/'Demo.app'/'Contents'/'Resources').mkdir(parents=True)

In [ ]:
st = write_stamp(repo/'dist'/'Demo.app', repo, version='2.0.1')
st['commit'][:12], st['dirty'], st['version']

('3b2bbb3a952a', False, '2.0.1')

In [ ]:
read_stamp(repo/'dist'/'Demo.app') == st

True

A tree with changes nobody committed stamps `dirty`, and the build prints that alongside the
commit. An app that misbehaves can then be traced to a tree that was never pushed.

In [ ]:
(repo/'x.txt').write_text('changed')
git_stamp(repo)['dirty']

True

In [ ]:
#| hide
t3 = TemporaryDirectory(); plain = Path(t3.name)
gone = plain/'dist'/'Demo'                               
test_eq(write_stamp(gone, repo, '2.0.1')['version'], '2.0.1')
assert not gone.exists(), 'no bundle, no stamp file'
test_is(read_stamp(gone), None)
stamp_path(plain).write_text('{ truncated')              
test_is(read_stamp(plain), None)

In [ ]:
#| export
def check(spec, root, venv=None):
    "What a build would do here, without doing it. Answers on any platform, Linux included."
    root, venv = Path(root), Path(venv or Path(root)/'packaging'/'.app-venv')
    rows = {'platform': sys.platform, 'interpreter': sys.executable,
            'app': spec.name, 'out': str(spec.out(root))}
    if sys.platform not in ('darwin', 'win32'):
        rows['freezer'] = f'none — {sys.platform} builds nothing; macOS and Windows build on their own OS'
    else: rows['freezer'] = 'py2app' if sys.platform == 'darwin' else 'py2exe'
    if sys.platform == 'darwin':
        rows['framework_build'] = is_framework()
        if not rows['framework_build']:
            rows['would_rebuild'] = str(venv)
            rows['on'] = str(framework_python() or '')
    try:
        import webview  # noqa: F401
        rows['pywebview'] = 'installed'
    except ImportError: rows['pywebview'] = 'MISSING'
    rows['running'] = running_from(spec.out(root))
    return rows

`check` reports build readiness without changing the environment or filesystem.

In [ ]:
demo = App(name='Demo', entry='demo_app.py', version='2.0.1')
rep = check(demo, repo)
{**rep, 'interpreter': '/proj/.venv/bin/python3', 'out': '/proj/dist/Demo'}

{'platform': 'linux',
 'interpreter': '/proj/.venv/bin/python3',
 'app': 'Demo',
 'out': '/proj/dist/Demo',
 'freezer': 'none — linux builds nothing; macOS and Windows build on their own OS',
 'pywebview': 'MISSING',
 'running': []}

In [ ]:
#| export
def build(spec, root, setup_py, venv=None, alias=False, force=False, rebuild_venv=False,
          identity=None):
    "Build `spec` into `root/dist`, using framework Python on macOS."
    root, venv = Path(root), Path(venv or Path(root)/'packaging'/'.app-venv')
    out = spec.out(root)
    if not force and (live := running_from(out)):
        raise SystemExit(
            f'{out} is running as pid {", ".join(map(str, live))}. Quit it first, or pass force.\n'
            'A build replaces the standard library the running app imports from, and it fails from '
            'then on with a zip error that says nothing about the build.')
    if sys.platform == 'darwin' and not is_framework():
        if os.environ.get(REEXEC):
            raise SystemExit(f'{sys.executable} is still not a framework build; run `check` to see why')
        if (found := framework_python()) is None:
            raise SystemExit('py2app needs a framework Python, and this one is a standalone build '
                             'whose stdlib extension modules are not files.\n'
                             'Install one and try again:  brew install python@3.12')
        print(f'framework Python: {found}')
        exe = build_venv(found, venv, root, spec.extras, force=rebuild_venv)
        run(exe, sys.argv[0], *(['--alias'] if alias else []), env={**os.environ, REEXEC: '1'})
        return out
    args = [sys.executable, str(setup_py)] + (['py2app'] if sys.platform == 'darwin' else [])
    if alias and sys.platform == 'darwin': args.append('--alias')
    run(*args, cwd=str(root))
    if out.exists() and sys.platform == 'darwin':
        for name, where in finish(out, spec, identity).items(): print(f'  {name}: {where}')
    st = write_stamp(out, root, spec.version)
    print(f"\nbuilt {out}\n  from {st['commit'][:12]}"
          f"{' with uncommitted changes' if st['dirty'] else ''}")
    return out

`build` runs the platform freezer in an isolated environment. macOS re-execs once under a compatible framework interpreter when required.

In [ ]:
#| hide
for t in (tmp, t2, t3): t.cleanup()